# 06 · Deploy the web app (Hugging Face Space)

Notebook 04 normally deploys the app at the end. Use this notebook to (re)deploy it on its own. It needs **no GPU**, so the default CPU runtime is fine, and it takes about 2 minutes.

It will:
1. log in to Hugging Face with your **write** token (https://huggingface.co/settings/tokens)
2. create the Gradio Space `<you>/kiswahili-sentiment-demo` and upload `app/app.py`, `app/requirements.txt`, `app/README.md` and `src/preprocessing.py`
3. point the app at your fine-tuned model `<you>/kiswahili-sentiment`
4. make sure the model card states the training loss that was actually used

In [ ]:
# --- Setup: works both locally and in Google Colab ---
import os, sys
REPO_URL = "https://github.com/Joellate/kiswahili-sentiment-analysis.git"

if "google.colab" in sys.modules:
    if not os.path.exists("kiswahili-sentiment-analysis"):
        !git clone -q $REPO_URL
    %cd kiswahili-sentiment-analysis
    %pip install -q -r requirements.txt
else:
    # running locally from notebooks/
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")

sys.path.insert(0, "src")
print("Working directory:", os.getcwd())

In [ ]:
from getpass import getpass
from huggingface_hub import CommitOperationAdd, HfApi, hf_hub_download, login, whoami

login(token=getpass("Paste your Hugging Face WRITE token and press Enter: ").strip())
HF_USER = whoami()["name"]
MODEL_REPO = f"{HF_USER}/kiswahili-sentiment"
SPACE_REPO = f"{HF_USER}/kiswahili-sentiment-demo"
api = HfApi()
print("Logged in as", HF_USER)
print("Model exists:", api.repo_exists(MODEL_REPO))

## Create the Space and upload the app

In [ ]:
try:
    api.create_repo(SPACE_REPO, repo_type="space", space_sdk="gradio", exist_ok=True)
    print("Space ready:", SPACE_REPO)
except Exception as e:
    print("Could not create the Space:", repr(e))
    raise

files = {"app/app.py": "app.py", "app/requirements.txt": "requirements.txt",
         "app/README.md": "README.md", "src/preprocessing.py": "preprocessing.py"}
api.create_commit(repo_id=SPACE_REPO, repo_type="space", commit_message="Deploy Gradio app",
                  operations=[CommitOperationAdd(path_in_repo=r, path_or_fileobj=l) for l, r in files.items()])
print("Uploaded:", list(files.values()))

try:
    api.add_space_variable(SPACE_REPO, "MODEL_ID", MODEL_REPO)
    print("MODEL_ID =", MODEL_REPO)
except Exception as e:
    print("Could not set MODEL_ID (the app falls back to its default):", repr(e))

## Make the model card match the training run

The card template written by notebook 04 always said "class-weighted cross-entropy". If the selected run was the *without class weights* ablation, this cell corrects the text.

In [ ]:
card_path = hf_hub_download(MODEL_REPO, "README.md")
card = open(card_path, encoding="utf-8").read()
if "without class weights" in card and "class-weighted cross-entropy" in card:
    card = card.replace("class-weighted cross-entropy", "unweighted cross-entropy (no class weights)")
    api.upload_file(path_or_fileobj=card.encode("utf-8"), path_in_repo="README.md", repo_id=MODEL_REPO,
                    commit_message="Model card: correct training loss description")
    print("Model card corrected")
else:
    print("Model card already correct")

## Check the build

The first build takes about 3–5 minutes. Re-run this cell until the stage is `RUNNING`, then open the link.

In [ ]:
rt = api.get_space_runtime(SPACE_REPO)
print("Stage:", rt.stage)
print("Open the app: https://huggingface.co/spaces/" + SPACE_REPO)
if rt.stage in ("BUILD_ERROR", "RUNTIME_ERROR"):
    print("Something failed. Open the link above and click 'Logs' to see why.")